In [ ]:
# First cell - All imports grouped together
import numpy as np
import pandas as pd
import os
import random
import matplotlib.pyplot as plt

# sklearn imports
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# ART imports
from art.estimators.classification import SklearnClassifier
from art.attacks.evasion import HopSkipJump
from art.attacks.poisoning import backdoor_attack

# Optional XGBoost
import xgboost as xgb


C:\Users\Jan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Jan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\art\estimators\certification\__init__.py:30: UserWarning: PyTorch not found. Not importing DeepZ or Interval Bound Propagation functionality
  warnings.warn("PyTorch not found. Not importing DeepZ or Interval Bound Propagation functionality")


ImportError: cannot import name 'PoisoningAttackLabelFlipping' from 'art.attacks.poisoning' (C:\Users\Jan\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\art\attacks\poisoning\__init__.py)

In [ ]:
# set seeds for reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
def debug_shapes(**arrays):
    for name, arr in arrays.items():
        try:
            print(f"{name}: shape={np.asarray(arr).shape}, dtype={np.asarray(arr).dtype}")
        except Exception as e:
            print(f"{name}: could not inspect ({e})")

In [ ]:
def label_flip_poison(X_train, y_train, flip_fraction=0.05, rng_seed=0, flip_only_from=None):
    rng = np.random.RandomState(rng_seed)
    y_poison = np.array(y_train).copy().astype(int)
    n = len(y_poison)
    if flip_only_from is None:
        # choose indices uniformly
        n_flip = int(np.ceil(n * flip_fraction))
        flip_idx = rng.choice(np.arange(n), size=n_flip, replace=False)
    else:
        src_idx = np.where(y_poison == flip_only_from)[0]
        n_flip = int(np.ceil(len(src_idx) * flip_fraction))
        flip_idx = rng.choice(src_idx, size=n_flip, replace=False)
    # binary flip (0<->1)
    y_poison[flip_idx] = 1 - y_poison[flip_idx]
    return X_train.copy(), y_poison

In [ ]:
# Then run the data loading and splitting code in next cell
df = pd.read_csv("C:\\Users\\Jan\\Git\\AnomalyDetection\\CSVs\\dataset.csv")
y = df["anomaly"]
X = df.drop(columns=["anomaly", "timestamp", "channel", "label", "train"], errors="ignore")

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

In [ ]:
# 2) Prepare X, y
# (Already done above)

In [ ]:
print('Loaded X, y')
print('X shape:', X.shape, 'y shape:', y.shape)

Loaded X, y
X shape: (2123, 20) y shape: (2123,)


In [ ]:
# Scale (use same scaler for all models and for surrogates)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype(np.float32)
X_test_s = scaler.transform(X_test).astype(np.float32)


# clip_values: per-feature min/max on the *scaled* training set
X_min = X_train_s.min(axis=0).astype(np.float32)
X_max = X_train_s.max(axis=0).astype(np.float32)
clip_values = (X_min, X_max)


print('Preprocessing done')
debug_shapes(X_train_s=X_train_s, X_test_s=X_test_s, y_train=y_train, y_test=y_test)

Preprocessing done
X_train_s: shape=(1698, 20), dtype=float32
X_test_s: shape=(425, 20), dtype=float32
y_train: shape=(1698,), dtype=int64
y_test: shape=(425,), dtype=int64


In [ ]:
models = {
"SVM": SVC(kernel='linear', C=1.0, probability=True, random_state=100),
"NeuralNet": MLPClassifier(hidden_layer_sizes=(128,64), activation='relu', solver='adam', alpha=0.0001,
batch_size=64, learning_rate='adaptive', learning_rate_init=0.01,
max_iter=1000, early_stopping=True, n_iter_no_change=20, random_state=100),
"LogisticRegression": LogisticRegression(solver='liblinear', penalty='l2', C=10,
class_weight='balanced', random_state=100),
#"XGBoost": xgb.XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=4,
#min_child_weight=1, tree_method='hist', eval_metric='logloss',
#random_state=100, n_jobs=-1),
"RandomForest": RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_split=5,
class_weight='balanced', min_samples_leaf=1, random_state=100, n_jobs=-1)

}


In [ ]:
trained_models = {}
for name, m in models.items():
    print('Training', name)
    m_clone = clone(m)
    m_clone.fit(X_train_s, y_train)
    trained_models[name] = m_clone
    preds = m_clone.predict(X_test_s)
    print(f"{name} clean accuracy: {accuracy_score(y_test, preds):.4f}")

Training SVM
SVM clean accuracy: 0.9247
Training NeuralNet
NeuralNet clean accuracy: 0.9576
Training LogisticRegression
LogisticRegression clean accuracy: 0.9271
Training RandomForest
RandomForest clean accuracy: 0.9506


In [ ]:
from art.estimators.classification import SklearnClassifier
from art.attacks.evasion import HopSkipJump

In [ ]:
hsj_kwargs = dict(max_iter=10, max_eval=50, init_eval=10, targeted=False)


results = {}
for name, model in trained_models.items():
    print('\n=== Attacking model:', name)
    art_clf = SklearnClassifier(model=model, clip_values=clip_values)
    attack = HopSkipJump(classifier=art_clf, **hsj_kwargs)
    # generate adversarials (may take time)
    X_test_adv = attack.generate(x=X_test_s)
    pred_clean = model.predict(X_test_s)
    pred_adv = model.predict(X_test_adv)
    clean_acc = (pred_clean == y_test).mean()
    adv_acc = (pred_adv == y_test).mean()
    print(f"{name} clean acc: {clean_acc:.4f}, adv acc (HopSkipJump): {adv_acc:.4f}")
    results[name] = dict(clean_acc=clean_acc, adv_acc=adv_acc)


=== Attacking model: SVM


HopSkipJump: 100%|██████████| 425/425 [00:04<00:00, 86.18it/s]


SVM clean acc: 0.9247, adv acc (HopSkipJump): 0.1765

=== Attacking model: NeuralNet


HopSkipJump: 100%|██████████| 425/425 [00:04<00:00, 91.70it/s]


NeuralNet clean acc: 0.9576, adv acc (HopSkipJump): 0.0565

=== Attacking model: LogisticRegression


HopSkipJump: 100%|██████████| 425/425 [00:04<00:00, 103.96it/s]


LogisticRegression clean acc: 0.9271, adv acc (HopSkipJump): 0.1812

=== Attacking model: RandomForest


HopSkipJump:   0%|          | 1/425 [00:04<28:59,  4.10s/it]


KeyboardInterrupt: 

In [ ]:
from art.attacks.poisoning import PoisoningAttackBackdoor as LabelFlipAttack
from sklearn.base import clone

ModuleNotFoundError: No module named 'art.attacks.poisoning.poisoning_attack_label_flipping'

In [3]:
def label_flip_art(X_train, y_train, classifier, flip_ratio, flip_labels):
    # Convert the sklearn classifier to ART classifier
    art_classifier = SklearnClassifier(model=classifier)
    
    # Initialize Label Flip attack with correct parameters
    # The first label in flip_labels is source, second is target
    attack = LabelFlipAttack(classifier=art_classifier, 
                                         target_label=flip_labels[1][1],
                                         source_label=flip_labels[0][0])
    
    # Generate poisoned training set
    x_poisoned, y_poisoned = attack.poison(X_train, y_train)
    
    # Retrain on poisoned data
    classifier.fit(x_poisoned, y_poisoned)
    
    # Calculate accuracy on test set
    y_pred = classifier.predict(X_test_s)
    poisoned_acc = accuracy_score(y_test, y_pred)
    
    return x_poisoned, y_poisoned, classifier, poisoned_acc

In [4]:
flip_rates = [0.03, 0.05, 0.1, 0.15, 0.20]
poison_results = {}
n_poison = [(0, 1), (1, 0)]

for flip in flip_rates:
    print(f"\n--- Label-flip poisoning: flip_fraction={flip}")
    poison_results[flip] = {}
    for name, classifier in models.items():
        print(' Retraining', name)
        classifier_clone = clone(classifier)
        X_pois, y_pois, clone, acc = label_flip_art(X_train_s, y_train, classifier_clone, flip, n_poison)
        poison_results[flip][name] = acc
        print(f" {name} acc after poison: {acc:.4f}")


--- Label-flip poisoning: flip_fraction=0.03


NameError: name 'models' is not defined

In [ ]:
# Quick summary table
import pandas as pd
poison_df = pd.DataFrame(poison_results).T
print('\nLabel-flip results (columns = models):')
print(poison_df)